In [115]:
import numpy as np

<img src="https://res.cloudinary.com/dnmobechs/image/upload/v1784908303/ChatGPT_Image_Jul_24_2026_09_18_53_PM_r4eiaj.png" />

In [116]:
class Tensor:
    def __init__(self, data, parents=(), op=""):
        # Ensure data is a NumPy array, converting if necessary.
        if not isinstance(data, np.ndarray):
            data = np.array(data)
        # Store the data as a float32 NumPy array for consistency.
        self.data = data.astype(np.float32)
        # Initialize gradient with zeros, matching the shape of the data.
        self.grad = np.zeros_like(self.data)
        # Store the operation that created this tensor (e.g., '+', '*', '@').
        self.op = op
        # Keep track of parent tensors for building the computational graph.
        self.parents = parents
        # Initialize backward function, which will be set by operations.
        self._backward = lambda: None

    # String representation for easy printing of Tensor objects.
    def __repr__(self):
        return f"Tensor(shape={self.data.shape}, op='{self.op}')"

    # Defines element-wise addition for two tensors.
    def __add__(self, other):
        """
        Add two tensors elementwise.

        Example:
            a = Tensor([1, 2, 3])
            b = Tensor([4, 5, 6])
            c = a + b  # c.data = [5, 7, 9]

        Broadcasting:
            a = Tensor([[1, 2], [3, 4]])  # shape (2, 2)
            b = Tensor([10, 20])          # shape (2,)
            c = a + b  # b broadcasts to (2, 2)
                       # c = [[11, 22], [13, 24]]
        """
        # Convert 'other' to a Tensor if it's not already one.
        other = other if isinstance(other, Tensor) else Tensor(other)
        # Perform element-wise addition to get the output data.
        out = Tensor(
            self.data + other.data,
            # Link parent tensors for backpropagation.
            parents=(self, other),
            op="+"
        )

        # Define the backward pass for addition.
        def _backward():
            # If 'self' was broadcasted during the forward pass, sum gradients over the broadcasted axes.
            if self.data.shape != out.grad.shape:
                self.grad += self._sum_broadcast(out.grad, self.data.shape)
            else:
                self.grad += out.grad # Otherwise, directly add the output gradient.
            # If 'other' was broadcasted during the forward pass, sum gradients over the broadcasted axes.
            if other.data.shape != out.grad.shape:
                other.grad += self._sum_broadcast(out.grad, other.data.shape)
            else:
                other.grad += out.grad # Otherwise, directly add the output gradient.

        out._backward = _backward
        return out

    # Helper function to correctly sum gradients over broadcasted dimensions.
    def _sum_broadcast(self, grad, target_shape):
        if grad.shape == target_shape:
            return grad # No broadcasting occurred, return gradient as is.

        ndims = len(grad.shape)
        # Pad the target shape with 1s to match the number of dimensions of the gradient for comparison.
        target_padded = (1,) * (ndims - len(target_shape)) + target_shape

        axes_to_sum = []
        # Identify axes along which broadcasting occurred (where dimensions don't match).
        for i, (g_dim, t_dim) in enumerate(zip(grad.shape, target_padded)):
            if g_dim != t_dim:
                axes_to_sum.append(i)

        # If broadcasted axes are found, sum the gradient along these axes.
        if axes_to_sum:
            grad = grad.sum(axis=tuple(axes_to_sum), keepdims=False)

        # Reshape the gradient to match the target shape.
        return grad.reshape(target_shape)

    # Defines element-wise multiplication for two tensors.
    def __mul__(self, other):
        """
        Multiply two tensors elementwise.

        Example:
            a = Tensor([1, 2, 3])
            b = Tensor([4, 5, 6])
            c = a * b  # c.data = [4, 10, 18]

        Gradient: d(a*b)/da = b, d(a*b)/db = a
        """
        # Convert 'other' to a Tensor if it's not already one.
        other = other if isinstance(other, Tensor) else Tensor(other)

        # Perform element-wise multiplication.
        out = Tensor(
            self.data * other.data,
            parents=(self, other),
            op="*"
        )

        # Define the backward pass for multiplication.
        def _backward():
            # Gradient of multiplication:
            # dL/da = dL/d(a*b) * b
            # dL/db = dL/d(a*b) * a

            # Calculate gradient for 'self'.
            self_grad = out.grad * other.data
            # Apply sum_broadcast if 'self' was broadcasted.
            if self.data.shape != self_grad.shape:
                self.grad += self._sum_broadcast(self_grad, self.data.shape)
            else:
                self.grad += self_grad

            # Calculate gradient for 'other'.
            other_grad = out.grad * self.data
            # Apply sum_broadcast if 'other' was broadcasted.
            if other.data.shape != other_grad.shape:
                other.grad += self._sum_broadcast(other_grad, other.data.shape)
            else:
                other.grad += other_grad

        out._backward = _backward
        return out

    # Defines matrix multiplication for two tensors.
    def __matmul__(self, other):
        """
        Matrix multiplication: self @ other

        Forward: C = A @ B
        - A.shape = (batch, in_features)
        - B.shape = (in_features, out_features)
        - C.shape = (batch, out_features)

        Backward:
        - dC/dA = dC @ B.T
        - dC/dB = A.T @ dC

        Important: The gradients use matrix multiplication with transposes!
        """
        # Convert 'other' to a Tensor if it's not already one.
        other = other if isinstance(other, Tensor) else Tensor(other)

        # Forward pass: perform matrix multiplication.
        out = Tensor(
            self.data @ other.data,
            parents=(self, other),
            op="@"
        )

        # Define the backward pass for matrix multiplication.
        def _backward():
            # Gradient with respect to self (A):
            # If C = A @ B, then dL/dA = dL/dC @ B.T (where dL/dC is out.grad)
            self_grad = out.grad @ other.data.T

            # Handle potential broadcasting (though less common in explicit matmul).
            if self.data.shape != self_grad.shape:
                self.grad += self._sum_broadcast(self_grad, self.data.shape)
            else:
                self.grad += self_grad

            # Gradient with respect to other (B):
            # If C = A @ B, then dL/dB = A.T @ dL/dC
            other_grad = self.data.T @ out.grad

            # Handle potential broadcasting.
            if other.data.shape != other_grad.shape:
                other.grad += self._sum_broadcast(other_grad, other.data.shape)
            else:
                other.grad += other_grad

        out._backward = _backward
        return out

    # Reduces the tensor to a scalar by summing all its elements.
    def sum(self):
      # Create a new Tensor with the sum of data and link 'self' as a parent.
      out = Tensor(self.data.sum(), parents=(self,))
      # Define the backward pass for the sum operation.
      def _backward():
        # The gradient of sum is simply the gradient of the output propagated to all elements, scaled by ones.
        self.grad += np.ones_like(self.data) * out.grad
      out._backward = _backward
      return out


    # Initiates the backpropagation process to compute gradients.
    def backward(self):
        # Initialize data structures for topological sort.
        topo = []
        visited = set()

        # Helper function to build the topological order of the computational graph.
        def build_topo(v):
            """Recursively collect all nodes in the graph in topological order."""
            if v not in visited:
                visited.add(v)
                for p in v.parents:
                    build_topo(p)
                topo.append(v)

        # Start building the graph from the current tensor.
        build_topo(self)

        # Initialize the gradient of the output tensor (dL/dL = 1).
        # For a scalar output, grad is 1.0; for a vector/matrix, it's ones_like.
        if self.data.size == 1:
            self.grad = np.array([1.0])
        else:
            self.grad = np.ones_like(self.data)

        # Iterate through the graph in reverse topological order and call each tensor's _backward method.
        for v in reversed(topo):
            v._backward()

        print(f"Backward complete! Processed {len(topo)} nodes.")

In [117]:
# Created some tensors with different data types and shapes.
# t1 is a 1D tensor (vector).
t1 = Tensor([1, 2, 3])
# t2 is a 2D tensor (matrix).
t2 = Tensor(np.array([[1, 2], [3, 4]]))
# t3 is a 0D tensor (scalar).
t3 = Tensor(5.0)

# Print the tensor objects themselves.
print(t1)
print(t2)
print(t3)

# Display the shapes of the data stored within each tensor.
print("\nData shapes:")
print("t1.data:", t1.data.shape)
print("t2.data:", t2.data.shape)
print("t3.data:", t3.data.shape)

# Display the initial gradients (all zeros) for each tensor.
print("\nGradients (all zero):")
print("t1.grad:", t1.grad)
print("t2.grad:", t2.grad)
print("t3.grad:", t3.grad)

Tensor(shape=(3,), op='')
Tensor(shape=(2, 2), op='')
Tensor(shape=(), op='')

Data shapes:
t1.data: (3,)
t2.data: (2, 2)
t3.data: ()

Gradients (all zero):
t1.grad: [0. 0. 0.]
t2.grad: [[0. 0.]
 [0. 0.]]
t3.grad: 0.0


<img src="https://res.cloudinary.com/dnmobechs/image/upload/v1784908668/ChatGPT_Image_Jul_24_2026_09_27_35_PM_g3k2g4.png" />

In [118]:
print("Matrix Multiplication Gradients")
# This section demonstrates matrix multiplication and its backward pass.

# Example with shapes for matrix multiplication.
batch = 4
in_features = 3
out_features = 2

# Create two random tensors for matrix multiplication.
A = Tensor(np.random.randn(batch, in_features))
B = Tensor(np.random.randn(in_features, out_features))

# Print the shapes of the input tensors.
print(f"A.shape: {A.data.shape}")
print(f"B.shape: {B.data.shape}")
# Perform matrix multiplication and print the resulting shape.
print(f"C = A @ B shape: {(A @ B).data.shape}")

# Demonstrate the transpose logic which is crucial for matrix multiplication gradients.
print("\nManual transpose check:")
print(f"B.T.shape: {B.data.T.shape}")
print(f"A.T.shape: {A.data.T.shape}")

# --- Example for element-wise operations and sum backward ---
# Re-initialize A and B with simpler 1D tensors for a clear gradient example.
A = Tensor([2.0, 3.0, 4.0])
B = Tensor([1.0, 2.0, 3.0])

# Define a simple computation: (A * B + A).sum()
# L = (A_i * B_i + A_i) for each element, then sum all results.
# dL/dA_i = B_i + 1
# dL/dB_i = A_i
L = (A * B + A).sum()

# Print the final scalar value of L and its shape.
print(f"\n\nL.data: {L.data}")
print(f"L.shape: {L.data.shape}")

# Perform automatic backward pass to compute gradients.
L.backward()
print("\n\nGradients after backward:")
# For A, expected gradient is B + 1 element-wise.
print(f"a.grad: {A.grad}")
# For B, expected gradient is A element-wise.
print(f"b.grad: {B.grad}")

print("\nExpected:")
print(f"a.grad expected: {B.data + 1.0}")
print(f"b.grad expected: {A.data}")

Matrix Multiplication Gradients
A.shape: (4, 3)
B.shape: (3, 2)
C = A @ B shape: (4, 2)

Manual transpose check:
B.T.shape: (2, 3)
A.T.shape: (3, 4)


L.data: 29.0
L.shape: ()
Backward complete! Processed 5 nodes.


Gradients after backward:
a.grad: [2. 3. 4.]
b.grad: [2. 3. 4.]

Expected:
a.grad expected: [2. 3. 4.]
b.grad expected: [2. 3. 4.]


<img src="https://res.cloudinary.com/dnmobechs/image/upload/v1784908837/ChatGPT_Image_Jul_24_2026_09_30_25_PM_ttdcn7.png" />

In [119]:
# --- Test Cases for Tensor Operations ---

# Test 1: Simple element-wise addition of two 1D tensors.
print("Test 1: Simple Addition")
a = Tensor([1, 2, 3])
b = Tensor([4, 5, 6])
c = a + b # Result: [5, 7, 9]

print(f"a.data: {a.data}")
print(f"b.data: {b.data}")
print(f"c.data: {c.data}") # Expected: [5. 7. 9.]
print(f"c.op: {c.op}") # Expected: +
print(f"c.parents: {c.parents}") # Expected: (Tensor(...), Tensor(...))

# Test 2: Broadcasting addition, where a 1D tensor is added to a 2D tensor.
print("\nTest 2: Broadcasting Addition")
a = Tensor([[1, 2], [3, 4]])  # shape (2, 2)
b = Tensor([10, 20])           # shape (2,) - will be broadcasted to [[10, 20], [10, 20]]
c = a + b

print(f"a.shape: {a.data.shape}")
print(f"b.shape: {b.data.shape}")
print(f"c.shape: {c.data.shape}") # Expected: (2, 2)
print(f"c.data:\n{c.data}") # Expected: [[11, 22], [13, 24]]
print(f"c.op: {c.op}") # Expected: +


# Test 1: Simple element-wise multiplication of two 1D tensors.
print("\n\nTest 1: Simple Multiplication")
a = Tensor([1, 2, 3])
b = Tensor([4, 5, 6])
c = a * b # Result: [4, 10, 18]

print(f"a.data: {a.data}")
print(f"b.data: {b.data}")
print(f"c.data: {c.data}") # Expected: [ 4. 10. 18.]
print(f"Manual: {a.data * b.data}") # Verify manually computed result
print(f"Match? {np.allclose(c.data, a.data * b.data)}") # Check if results match

# Test 2: Broadcasting multiplication, where a 1D tensor is multiplied with a 2D tensor.
print("\nTest 2: Broadcasting Multiplication")
a = Tensor([[1, 2], [3, 4]])  # shape (2, 2)
b = Tensor([2, 3])             # shape (2,) - will be broadcasted to [[2, 3], [2, 3]]
c = a * b

print(f"a.shape: {a.data.shape}")
print(f"b.shape: {b.data.shape}")
print(f"c.shape: {c.data.shape}") # Expected: (2, 2)
print(f"c.data:\n{c.data}") # Expected: [[ 2.  6.], [ 6. 12.]]
print(f"Manual:\n{a.data * b.data}") # Verify manually computed result
print(f"Match? {np.allclose(c.data, a.data * b.data)}") # Check if results match

Test 1: Simple Addition
a.data: [1. 2. 3.]
b.data: [4. 5. 6.]
c.data: [5. 7. 9.]
c.op: +
c.parents: (Tensor(shape=(3,), op=''), Tensor(shape=(3,), op=''))

Test 2: Broadcasting Addition
a.shape: (2, 2)
b.shape: (2,)
c.shape: (2, 2)
c.data:
[[11. 22.]
 [13. 24.]]
c.op: +


Test 1: Simple Multiplication
a.data: [1. 2. 3.]
b.data: [4. 5. 6.]
c.data: [ 4. 10. 18.]
Manual: [ 4. 10. 18.]
Match? True

Test 2: Broadcasting Multiplication
a.shape: (2, 2)
b.shape: (2,)
c.shape: (2, 2)
c.data:
[[ 2.  6.]
 [ 6. 12.]]
Manual:
[[ 2.  6.]
 [ 6. 12.]]
Match? True
